# Box plots
Generates boxplot figures comparing model performance across feature strategies, CV schemes, and experiment types. Execute cells top to bottom. Only cells marked **[CONFIGURE]** require changes.

> Run `run_experiments.ipynb` before running this notebook.

## 1. Environment Setup **[OPTIONAL]**
Mounts Google Drive and installs dependencies when running on Colab. Skip if running locally.

> **NOTE: requires moving `PROJECT` folder to Google Drive.**

In [ ]:
# COLAB
import sys
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    %cd '/content/drive/My Drive/PROJECT'

    !pip install -q numpy matplotlib seaborn

    sys.path.insert(0, './product/src')

## 2. Imports

In [ ]:
import os
import sys
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker
import seaborn as sns
from pathlib import Path
from utils.paths import get_project_root
from visualisation.common import discover_timestamps, auc_from_roc
from visualisation.box_plots import metrics_from_cm, extract_metrics, plot_metric_figure
from data_io.centralised_logger import CentralisedLogger

## 3. Configuration **[CONFIGURE]**

The `RUNS` table maps each `(model, dim)` combination to its logger `folder_name` and a `tags` dict. Each tag assigns a `'CV-Experiment'` label to a timestamp discovered in that folder. Valid tag values are: `'SKF-Baseline'`, `'SKF-Optimised'`, `'LOSO-Baseline'`, `'LOSO-Optimised'`.

Use `FILTER` to restrict which configurations are loaded and plotted. Set any entry to `None` to include everything for that dimension.

| Column | Description |
|---|---|
| `folder_name` | The `model_name` string passed to `CentralisedLogger` |
| `tags` | Maps timestamp index (0 = oldest) to a `'CV-Experiment'` label |

In [ ]:
# ---- CHANGE THESE ------------------------------------------------
# (model, dim) -> (folder_name, tags)
# tags: {timestamp_index: 'CV-Experiment'}
RUNS = {
    ('svm', 'selectkbest'): ('svm_selectkbest', {0: 'SKF-Baseline', 1: 'LOSO-Baseline', 2: 'SKF-Optimised', 3: 'LOSO-Optimised'}),
    ('svm', 'pca'):         ('svm_pca',         {0: 'SKF-Baseline', 1: 'LOSO-Baseline', 2: 'SKF-Optimised', 3: 'LOSO-Optimised'}),
    ('svm', 'sdae'):        ('svm_sdae',         {0: 'SKF-Baseline', 1: 'LOSO-Baseline', 2: 'SKF-Optimised', 3: 'LOSO-Optimised'}),
    ('rf',  'selectkbest'): ('rf_selectkbest',  {0: 'SKF-Baseline', 1: 'LOSO-Baseline', 2: 'SKF-Optimised', 3: 'LOSO-Optimised'}),
    ('rf',  'pca'):         ('rf_pca',           {0: 'SKF-Baseline', 1: 'LOSO-Baseline', 2: 'SKF-Optimised', 3: 'LOSO-Optimised'}),
    ('rf',  'sdae'):        ('rf_sdae',           {0: 'SKF-Baseline', 1: 'LOSO-Baseline', 2: 'SKF-Optimised', 3: 'LOSO-Optimised'}),
    ('xgb', 'selectkbest'): ('xgb_selectkbest', {0: 'SKF-Baseline', 1: 'LOSO-Baseline', 2: 'SKF-Optimised', 3: 'LOSO-Optimised'}),
    ('xgb', 'pca'):         ('xgb_pca',          {0: 'SKF-Baseline', 1: 'LOSO-Baseline', 2: 'SKF-Optimised', 3: 'LOSO-Optimised'}),
    ('xgb', 'sdae'):        ('xgb_sdae',          {0: 'SKF-Baseline', 1: 'LOSO-Baseline', 2: 'SKF-Optimised', 3: 'LOSO-Optimised'}),
    ('gcn', 'selectkbest'): ('gcn_selectkbest', {0: 'SKF-Baseline', 1: 'LOSO-Baseline', 2: 'SKF-Optimised', 3: 'LOSO-Optimised'}),
    ('gcn', 'pca'):         ('gcn_pca',          {0: 'SKF-Baseline', 1: 'LOSO-Baseline', 2: 'SKF-Optimised', 3: 'LOSO-Optimised'}),
    ('gcn', 'sdae'):        ('gcn_sdae',          {0: 'SKF-Baseline', 1: 'LOSO-Baseline', 2: 'SKF-Optimised', 3: 'LOSO-Optimised'}),
}

# Filter which configurations to load and plot.
# Set any entry to None to include all options for that dimension.
FILTER = {
    'experiments': ['Baseline', 'Optimised'],     # None or e.g. ['Baseline']
    'cv':          ['SKF', 'LOSO'],               # None or e.g. ['SKF']
    'metrics':     ['auc'],                       # None or e.g. ['auc', 'sens']
    'models':      ['svm'],                       # None or e.g. ['svm', 'xgb']
    'dims':        ['selectkbest'],               # None or e.g. ['selectkbest']
}
# ------------------------------------------------------------------

## 4. Discover Timestamps **[CONFIGURE]**

Scans each logger folder and prints all discovered timestamps with their index.

In [ ]:
# Print discovered timestamps for every folder in RUNS
seen_folders = set()
for (model, dim), (folder_name, tags) in RUNS.items():
    if folder_name in seen_folders:
        continue
    seen_folders.add(folder_name)
    timestamps = discover_timestamps(folder_name)
    print(f"{folder_name}")
    if not timestamps:
        print("  (no runs found)")
    for i, ts in enumerate(timestamps):
        current_tag = next(
            (t for (m, d), (fn, tgs) in RUNS.items()
             if fn == folder_name for idx, t in tgs.items() if idx == i),
            '(untagged)'
        )
        print(f"  [{i}]  {ts}  ->  {current_tag}")
    print()

## 5. Metric Extraction

Helper functions to compute AUC from stored ROC curves and derive accuracy, sensitivity, specificity, and F1 from confusion matrices.

## 6. Load All Data

Resolves timestamps from the `tags` in `RUNS`, applies `FILTER`, and loads per-fold metric lists into `all_data[experiment][cv][model][dim]`.

In [ ]:
ALL_MODELS      = ['svm', 'rf', 'xgb', 'gcn']
ALL_DIMS        = ['selectkbest', 'pca', 'sdae']
ALL_CV_SCHEMES  = ['SKF', 'LOSO']
ALL_EXPERIMENTS = ['Baseline', 'Optimised']
ALL_METRICS     = ['auc', 'acc', 'sens', 'spec', 'f1']
VALID_TAGS      = {'SKF-Baseline', 'SKF-Optimised', 'LOSO-Baseline', 'LOSO-Optimised'}

# Resolve FILTER: None means include all
MODELS      = FILTER['models']      or ALL_MODELS
DIMS        = FILTER['dims']        or ALL_DIMS
CV_SCHEMES  = FILTER['cv']          or ALL_CV_SCHEMES
EXPERIMENTS = FILTER['experiments'] or ALL_EXPERIMENTS
METRICS     = FILTER['metrics']     or ALL_METRICS

print("Running with:")
print(f"  experiments : {EXPERIMENTS}")
print(f"  cv          : {CV_SCHEMES}")
print(f"  metrics     : {METRICS}")
print(f"  models      : {MODELS}")
print(f"  dims        : {DIMS}")
print()

# all_data[experiment][cv][model][dim] = {'auc': [...], ...} or None
all_data = {
    exp: {cv: {m: {d: None for d in DIMS} for m in MODELS}
          for cv in CV_SCHEMES}
    for exp in EXPERIMENTS
}

for (model, dim), (folder_name, tags) in RUNS.items():
    if model not in MODELS or dim not in DIMS:
        continue

    timestamps = discover_timestamps(folder_name)

    for ts_idx, tag in tags.items():
        if tag not in VALID_TAGS:
            print(f"WARNING: unknown tag '{tag}' for {folder_name}[{ts_idx}] skipping")
            continue

        cv, exp = tag.split('-')

        if cv not in CV_SCHEMES or exp not in EXPERIMENTS:
            continue

        if ts_idx >= len(timestamps):
            print(f"WARNING: index {ts_idx} out of range for {folder_name} "
                  f"({len(timestamps)} runs found) skipping")
            continue

        timestamp     = timestamps[ts_idx]
        logger        = CentralisedLogger(model_name=folder_name)
        target_folder = logger.artefact_path / timestamp

        if not target_folder.exists():
            print(f"WARNING: folder not found {target_folder}")
            continue

        raw = logger.load_folder_artefacts(folder_path=target_folder)
        all_data[exp][cv][model][dim] = extract_metrics(raw)
        n_runs = len(raw)
        label  = 'sites' if cv == 'LOSO' else 'folds'
        print(f"Loaded {n_runs:>2} {label}  [{exp:>9}]  [{cv:>4}]  {model}  {dim}")

## 7. Plot Settings

Display labels and colours for models, metrics, and feature strategies. Edit `MODEL_COLOURS` or `METRIC_LABELS` here if needed.

In [ ]:
MODEL_LABELS = {'svm': 'SVM', 'rf': 'RF', 'xgb': 'XGB', 'gcn': 'GCN'}
muted_colors  = sns.color_palette('Set2', len(MODELS))
MODEL_COLOURS = dict(zip(MODELS, muted_colors))

METRIC_LABELS = {
    'auc':  'AUC-ROC',
    'acc':  'Accuracy',
    'sens': 'Sensitivity',
    'spec': 'Specificity',
    'f1':   'F1 Score',
}

DIM_LABELS = {
    'selectkbest': 'SelectKBest',
    'pca':         'PCA',
    'sdae':        'SDAE',
}

## 8. Plotting

Produces one figure per metric per CV scheme per experiment type. Each figure has panels per strategy with boxplots per model.

In [ ]:
for exp in EXPERIMENTS:
    for cv in CV_SCHEMES:
        for metric in METRICS:
            plot_metric_figure(
                experiment=exp, cv=cv, metric=metric,
                models=MODELS, dims=DIMS, all_data=all_data,
                model_colours=MODEL_COLOURS, metric_labels=METRIC_LABELS,
                dim_labels=DIM_LABELS, model_labels=MODEL_LABELS, save=True
            )